# 3B SAE ablation @ layer 27 (Colab)

Same protocol as 0.5B: ablate top-6 PEFT and top-6 S1 features in **var-CoT**.

**Runtime → T4 GPU**

Uploads:
1. `qwen3b-cot-sft-v2-minimal.zip` (browser OK — small)
2. `sae.pt` via **Google Drive** (not browser — ~256MB)


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography

In [ ]:
!pip install -q peft "transformers<4.50" "datasets<4" accelerate tqdm matplotlib
!pip uninstall -y torchao  # Colab ships 0.10; peft 0.19+ rejects it


In [ ]:
from pathlib import Path
import urllib.request

RAW = (
    "https://raw.githubusercontent.com/vladflorinfilip/"
    "Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography/main/"
)
need = Path("sparse_autoencoders/analyze_ablation.py")
if not need.exists():
    need.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW + "sparse_autoencoders/analyze_ablation.py", need)
    print("fetched", need)
else:
    print("have", need)
assert need.exists()


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

ADAPTER_DIR = "checkpoints/qwen3b-cot-sft-v2"
GENERATIONS = "data/evaluation_data/qwen/ETHICS/qwen3b_v2_critic.jsonl"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("Upload qwen3b-cot-sft-v2-minimal.zip")
uploaded = files.upload()
Path("checkpoints").mkdir(exist_ok=True)
with zipfile.ZipFile(next(iter(uploaded)), "r") as zf:
    zf.extractall("checkpoints")

root = Path("checkpoints")
if not (root / "qwen3b-cot-sft-v2" / "adapter_config.json").exists():
    if (root / "adapter_config.json").exists():
        dest = root / "qwen3b-cot-sft-v2"
        dest.mkdir(exist_ok=True)
        for name in ("adapter_config.json", "adapter_model.safetensors"):
            src = root / name
            if src.exists():
                src.rename(dest / name)

assert Path(ADAPTER_DIR, "adapter_model.safetensors").exists()
assert Path(GENERATIONS).exists()

# minimal zip has adapters only — tokenizer must come from base
if not Path(ADAPTER_DIR, "tokenizer_config.json").exists():
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(ADAPTER_DIR)
    print("saved tokenizer →", ADAPTER_DIR)

print("OK", ADAPTER_DIR)


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

LAYER = 27
D_MODEL = 2048
DICT_MULT = 8                  # match 0.5B expansion ratio
DICT_SIZE = D_MODEL * DICT_MULT  # 16384
TOP_K = 15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ART_DIR = Path(f"sparse_autoencoders/artifacts/ethics_3b_l{LAYER}")
FLIPS = Path("data/intervention_data/qwen/ETHICS/interventions/negated_minimal_cot_3b.jsonl")

def run(cmd):
    print("+", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"exit {p.returncode}")


print(f"layer={LAYER}/{36-1}  d_model={D_MODEL}  dict={DICT_SIZE} ({DICT_MULT}×)")
print(f"0.5B reference: L18/23, d_model=896, dict=7168 (8×)")
print(f"data: {GENERATIONS} (same ETHICS n=100 protocol as 0.5B)")

## Load SAE (`SKIP_TRAIN`)

Do **not** browser-upload the 236MB zip — it hangs/fails.

1. Put `sae.pt` on Google Drive (e.g. `MyDrive/ethics_3b_l27/sae.pt`)
2. Run the next cell (`USE_DRIVE = True`)

Top-6 feature IDs are hardcoded from your local `report.json`.

In [ ]:
SKIP_TRAIN = True
USE_DRIVE = True  # set False only for tiny test uploads
DRIVE_SAE = "/content/drive/MyDrive/ethics_3b_l27/sae.pt"  # change if needed

# from your local ethics_3b_l27/report.json
PEFT6 = [8413, 15754, 2403, 12753, 9711, 15074]
S1_6 = [10747, 2865, 10449, 8481, 8348, 11551]

from pathlib import Path
import shutil

ART_DIR = Path(f"sparse_autoencoders/artifacts/ethics_3b_l{LAYER}")
ART_DIR.mkdir(parents=True, exist_ok=True)

if SKIP_TRAIN:
    dest = ART_DIR / "sae.pt"
    if not dest.exists():
        if USE_DRIVE:
            from google.colab import drive
            drive.mount("/content/drive")
            src = Path(DRIVE_SAE)
            assert src.exists(), f"missing {src} — upload sae.pt to Drive and fix DRIVE_SAE"
            print(f"copy {src} → {dest}", flush=True)
            shutil.copy2(src, dest)
        else:
            from google.colab import files
            print("Upload sae.pt only (not the zip)")
            uploaded = files.upload()
            name = next(iter(uploaded))
            Path(name).replace(dest)
    assert dest.exists() and dest.stat().st_size > 1_000_000, dest

    # minimal report so later cells / ablate tooling have feature lists
    report = {
        "layer": LAYER,
        "dict_size": DICT_SIZE,
        "top_peft_minus_base": [{"feature": f, "score": 0.0} for f in PEFT6],
        "top_s1_flip_sensitive": [{"feature": f, "score": 0.0} for f in S1_6],
        "candidates": [],
    }
    (ART_DIR / "report.json").write_text(json.dumps(report, indent=2))
    print(f"OK {dest} ({dest.stat().st_size/1e6:.1f} MB)")
else:
    raise SystemExit("SKIP_TRAIN=False not used here — retrain in a fresh notebook")

print("top PEFT−base:", PEFT6)
print("top S1-flip:   ", S1_6)


## Ablate top-6 (var-CoT generate)

Same as 0.5B: greedy generation with SAE decoder-direction ablation.
Runs **two** jobs (no combined overlap on 3B):
- `top6_peft_generate` — fine-tuning-enriched units
- `top6_s1_flip_generate` — S1-flip-sensitive units


In [ ]:
import sys
from pathlib import Path

sae_dir = Path("sparse_autoencoders").resolve()
sys.path.insert(0, str(sae_dir))
sys.path.insert(0, str(Path("evaluation").resolve()))
from analyze_ablation import load_by_index, summarize_pair, print_summary

peft6 = list(PEFT6)
s1_6 = list(S1_6)

jobs = [
    ("top6_peft_generate", peft6),
    ("top6_s1_flip_generate", s1_6),
]

baseline = load_by_index(Path(GENERATIONS))
for name, features in jobs:
    out = ART_DIR / "ablations" / "var_cot" / f"{name}.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, "-u", str(sae_dir / "ablate_features.py"),
        "--features", *[str(f) for f in features],
        "--mode", "generate",
        "--artifact-dir", str(ART_DIR),
        "--model", ADAPTER_DIR,
        "--generations", GENERATIONS,
        "--output", str(out),
        "--device", DEVICE,
    ])
    print_summary(out.name, summarize_pair(baseline, load_by_index(out)))


## PEFT feature k-sweep (var-CoT)

Dose-response test: ablate the strongest `k = 1, 2, 6, 12, 15` PEFT−base-enriched SAE features. The completed top-6 PEFT run is reused rather than regenerated.

In [ ]:
# Run after the top-6 var-CoT PEFT ablation has completed.
PEFT_TOP15 = [8413, 15754, 2403, 12753, 9711, 15074, 9542, 12300, 7352, 15483, 14865, 8759, 7422, 11577, 5766]
K_VALUES = [1, 2, 6, 12, 15]

baseline = load_by_index(Path(GENERATIONS))
for k in K_VALUES:
    features = PEFT_TOP15[:k]
    # Reuse the completed top-6 run; other k values get their own output.
    if k == 6:
        out = ART_DIR / "ablations" / "var_cot" / "top6_peft_generate.jsonl"
    else:
        out = ART_DIR / "ablations" / "var_cot" / f"top{k}_peft_generate.jsonl"

    if not out.exists():
        run([
            sys.executable, "-u", str(sae_dir / "ablate_features.py"),
            "--features", *[str(f) for f in features],
            "--mode", "generate",
            "--artifact-dir", str(ART_DIR),
            "--model", ADAPTER_DIR,
            "--generations", GENERATIONS,
            "--output", str(out),
            "--device", DEVICE,
        ])
    print_summary(f"top-{k} PEFT features", summarize_pair(baseline, load_by_index(out)))

## Full-dictionary PEFT × S1 ranking

Computes both scores for **every SAE feature** (not an intersection of clipped top-15 lists), saves `report_full_combined.json`, and exposes `COMBINED6` for the causal ablation.

In [ ]:
# Full 16,384-feature ranking: |PEFT - base| × |original CoT - S1-flip CoT|
import gc
import json
import torch

sys.path.insert(0, "sparse_autoencoders")
from run_sae import encode, flip_pairs, last_token_acts, load_model, load_records, scoring_text, topk
from sae import SparseAutoencoder

RANK_BATCH_SIZE = 2
TOP_K_COMBINED = 15

# Load the saved base/PEFT activations; collect them only if absent.
acts_path = ART_DIR / "activations.pt"
records = load_records(GENERATIONS)
texts = [scoring_text(r) for r in records]

if acts_path.exists():
    acts = torch.load(acts_path, map_location="cpu")
    base_acts, peft_acts = acts["base_acts"], acts["peft_acts"]
    print("using saved base/PEFT activations", tuple(base_acts.shape))
else:
    print("collecting base activations…", flush=True)
    base_tok, base_model = load_model(BASE_MODEL, torch.device(DEVICE))
    base_acts = last_token_acts(base_model, base_tok, texts, LAYER, RANK_BATCH_SIZE)
    del base_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print("collecting PEFT activations…", flush=True)
    peft_tok, peft_model = load_model(ADAPTER_DIR, torch.device(DEVICE))
    peft_acts = last_token_acts(peft_model, peft_tok, texts, LAYER, RANK_BATCH_SIZE)
    torch.save(
        {"layer": LAYER, "base_acts": base_acts, "peft_acts": peft_acts,
         "records_path": GENERATIONS, "peft_model": ADAPTER_DIR},
        acts_path,
    )
    del peft_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# Load SAE and calculate every feature's base-vs-PEFT score.
sae_ckpt = torch.load(ART_DIR / "sae.pt", map_location="cpu")
sae = SparseAutoencoder(
    sae_ckpt["state_dict"]["encoder.weight"].shape[1],
    sae_ckpt["dict_size"],
)
sae.load_state_dict(sae_ckpt["state_dict"])
sae = sae.to(device=DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).eval()

base_z = encode(sae, base_acts, RANK_BATCH_SIZE)
peft_z = encode(sae, peft_acts, RANK_BATCH_SIZE)
peft_score = (peft_z - base_z).abs().mean(0)

# S1 sensitivity: run PEFT on original vs minimally S1-negated CoTs.
if not FLIPS.exists():
    run([sys.executable, "-u", "intervention/make_minimal_negations.py",
         "--generations", GENERATIONS, "--output", str(FLIPS)])
pairs = flip_pairs(records, str(FLIPS))
assert pairs, "no valid S1-flip pairs"
orig_texts = [f"{p['prompt']} {p['orig']}\nFinal answer:" for p in pairs]
flip_texts = [f"{p['prompt']} {p['flip']}\nFinal answer:" for p in pairs]

print(f"computing full S1 scores over {len(pairs)} flip pairs…", flush=True)
peft_tok, peft_model = load_model(ADAPTER_DIR, torch.device(DEVICE))
orig_z = encode(sae, last_token_acts(peft_model, peft_tok, orig_texts, LAYER, RANK_BATCH_SIZE), RANK_BATCH_SIZE)
flip_z = encode(sae, last_token_acts(peft_model, peft_tok, flip_texts, LAYER, RANK_BATCH_SIZE), RANK_BATCH_SIZE)
s1_score = (orig_z - flip_z).abs().mean(0)
del peft_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# The crucial difference from the earlier report: product over ALL dictionary units.
combined_score = peft_score * s1_score
combined_rows = [
    {"feature": int(i), "peft_minus_base": float(peft_score[i]),
     "flip": float(s1_score[i]), "combined": float(combined_score[i])}
    for i in torch.topk(combined_score, TOP_K_COMBINED).indices.tolist()
]

combined_report = {
    "layer": LAYER,
    "dict_size": int(sae_ckpt["dict_size"]),
    "n_examples": len(records),
    "n_flip_pairs": len(pairs),
    "ranking": "abs(peft_code - base_code) mean × abs(original_code - s1_flip_code) mean, full dictionary",
    "top_peft_minus_base": topk(peft_score, TOP_K_COMBINED),
    "top_s1_flip_sensitive": topk(s1_score, TOP_K_COMBINED),
    "candidates": combined_rows,
}
(ART_DIR / "report_full_combined.json").write_text(json.dumps(combined_report, indent=2))
torch.save(
    {"peft_score": peft_score, "s1_score": s1_score, "combined_score": combined_score},
    ART_DIR / "full_combined_scores.pt",
)

COMBINED6 = [row["feature"] for row in combined_rows[:6]]
print("True full-dictionary combined top-6:", COMBINED6)
for row in combined_rows[:6]:
    print(f"f{row['feature']:5d}  peft={row['peft_minus_base']:.3f}  s1={row['flip']:.3f}  product={row['combined']:.3f}")

## Causal test: ablate true full-combined top-6

Runs the same selected feature set under fixed-CoT (`score`) and var-CoT (`generate`). Run only after the full-dictionary ranking cell prints `COMBINED6`.

In [ ]:
assert "COMBINED6" in globals(), "run the full-dictionary ranking cell first"

sae_dir = Path("sparse_autoencoders").resolve()
sys.path.insert(0, str(sae_dir))
sys.path.insert(0, str(Path("evaluation").resolve()))
from analyze_ablation import load_by_index, summarize_pair, print_summary

baseline = load_by_index(Path(GENERATIONS))
for mode, folder, name in [
    ("score", "fixed_cot", "top6_full_combined_score"),
    ("generate", "var_cot", "top6_full_combined_generate"),
]:
    out = ART_DIR / "ablations" / folder / f"{name}.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, "-u", str(sae_dir / "ablate_features.py"),
        "--features", *[str(f) for f in COMBINED6],
        "--mode", mode,
        "--artifact-dir", str(ART_DIR),
        "--model", ADAPTER_DIR,
        "--generations", GENERATIONS,
        "--output", str(out),
        "--device", DEVICE,
    ])
    print_summary(out.name, summarize_pair(baseline, load_by_index(out)))

In [ ]:
# Matched Qwen 3B base-model control (no PEFT): run on the same first 100 ETHICS items.
# Colab GPU inference is substantially faster than local CPU/MPS after the one-time model download.
import os

BASE_3B = "Qwen/Qwen2.5-3B-Instruct"
BASE_3B_OUT = Path("data/evaluation_data/qwen/ETHICS/qwen3b_base.jsonl")
BASE_3B_OUT.parent.mkdir(parents=True, exist_ok=True)

run([
    sys.executable, "-u", "evaluation/evaluate_ethics_morality.py",
    "--model", BASE_3B,
    "--limit", "100",
    "--output", str(BASE_3B_OUT),
])

# The ablation chart's S1-follow metric should use the same critic as the PEFT
# baseline. Configure AZURE_OPENAI_* in Colab before running this section.
if os.getenv("AZURE_OPENAI_ENDPOINT") and os.getenv("AZURE_OPENAI_API_KEY") and os.getenv("AZURE_OPENAI_DEPLOYMENT"):
    run([
        sys.executable, "-u", "evaluation/score_cot_alignment.py",
        "--input", str(BASE_3B_OUT),
        "--task", "ethics",
        "--in-place",
    ])
else:
    print(
        "Base generations written to", BASE_3B_OUT,
        "\nSkipping critic: set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, and "
        "AZURE_OPENAI_DEPLOYMENT, then rerun only the score_cot_alignment.py command above."
    )

print("Use this later in the local plot command: --unadapted-base", BASE_3B_OUT)

In [ ]:
import shutil
from google.colab import files

zip_stem = f"sae_3b_l{LAYER}_artifacts"
shutil.make_archive(zip_stem, "zip", ART_DIR)
files.download(f"{zip_stem}.zip")
print("Downloaded", f"{zip_stem}.zip")